# 01 — Signatures and `Predict`

In this notebook, you will learn how to describe a language-model task in DSPy.

**What you'll learn**

You will learn how to:

- write a **signature** that names a task's inputs and outputs;
- run that signature with `dspy.Predict`;
- ask for Python types such as `bool`, `int`, and `list[str]`;
- read values from a `dspy.Prediction`; and
- use `dspy.inspect_history()` to see what DSPy sent to the model.

You do not need to know any of these terms yet. We will introduce them one at a time.

*Estimated time: about 20 minutes. Estimated cost: less than $0.02. Cached reruns are free.*


In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import dspy
from dotenv import load_dotenv              # pip install -r requirements.txt if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead
# STRONG_MODEL = "openrouter/moonshotai/kimi-k2.6"
# STRONG_MODEL = "openrouter/openai/gpt-4o"

lm = dspy.LM(MODEL, max_tokens=1000)        # cache=True by default: reruns are ~free
dspy.configure(lm=lm)

### Optional: view traces with Phoenix

Phoenix can show you a visual record, or **trace**, of each model call. Run the next cell if you want to use it, or skip the cell. The rest of the notebook works without Phoenix.

The cell finds or starts a local Phoenix server and prints a link. Notebook 11 explains tracing in more detail.


In [ ]:
# optional: Phoenix tracing (see notebook 11)
import obs

obs.enable_phoenix()

### Optional: view traces with MLflow

MLflow is another tool for recording model calls. Run the next cell if you want to try it, or skip it. You can also use Phoenix and MLflow together.

The traces are saved in this repository, and the cell prints a link to the MLflow interface. Notebook 11 explains the full setup.


In [ ]:
# optional: MLflow tracing (see notebook 11)
import obs

obs.enable_mlflow()
obs.mlflow_ui()            # prints the link where the traces appear

## Why use a signature?

Imagine that we want a model to answer a question. A hand-written prompt usually has to do three jobs:

1. explain the task;
2. tell the model how to format its reply; and
3. turn that reply into useful Python data.

DSPy separates these jobs. You start by writing a **signature**. A signature only says what information goes in and what information should come out.

Here is an example:

```text
context: list[str], question: str -> answer: str
```

Read the arrow as “use the values on the left to produce the values on the right.”

| Part | Meaning |
|---|---|
| `context: list[str]` | an input named `context`, containing a list of strings |
| `question: str` | an input named `question`, containing a string |
| `->` | separates inputs from outputs |
| `answer: str` | an output named `answer`, containing a string |

The names help explain the fields to the model. The types tell DSPy what Python values you expect. DSPy handles the prompt format and parses the reply for you. Later notebooks show how DSPy can also improve the instructions automatically.


## Your first `Predict`

`dspy.Predict(signature)` creates a small, callable DSPy module from a signature.

The signature below has one input, `sentence`, and one output, `sentiment`. The output type is `bool`, so its value should be either `True` or `False`.

When you call the module, pass each input by name, as in `sentence=...`. DSPy inputs use keyword arguments rather than positional arguments.


In [ ]:
clf = dspy.Predict("sentence -> sentiment: bool")
clf(sentence="This chowder is a crime against the sea.")

> **What you should see:** `Prediction(sentiment=False)`
>
> The exact model response can vary, but `sentiment` should be a Python `bool`, not the text `"False"`. DSPy asked the model for the format it needed and converted the response. You did not have to write parsing code.


A signature can have more than one output. It can also use different Python types.

In the next example, DSPy extracts a name, an age, and a list of hobbies from a short biography.


In [ ]:
extract = dspy.Predict("bio -> name: str, age: int, hobbies: list[str]")

pred = extract(bio="Odo, 71, retired lightkeeper, spends his days birdwatching and grinding lenses.")
print(pred.name, "|", pred.age, "|", pred.hobbies)

> **Expected result:** something similar to `Odo | 71 | ['birdwatching', 'grinding lenses']`
>
> Model wording can vary. Notice that `pred.age` is an `int` and `pred.hobbies` is a `list`, not a piece of JSON text.

`extract(...)` returns a `dspy.Prediction`. You can read each output as an attribute, such as `pred.name` or `pred.age`. Notebook 02 will show why `Prediction` objects are also useful to DSPy optimizers.


## See what the model received

DSPy created a prompt for the previous call even though we did not write one ourselves. Use `dspy.inspect_history()` when you want to see that prompt.


In [ ]:
dspy.inspect_history(n=1)

> **What to look for:** the last request and response.
>
> You should see a system message that explains the fields, a user message containing the biography, and the model's reply before DSPy parsed it. DSPy often uses markers such as `[[ ## name ## ]]` to separate fields.

This is a useful debugging habit: if a result surprises you, call `dspy.inspect_history(n=1)` to check what the model actually received. Notebook 02 explains how DSPy builds this prompt.


## Use a class when the task needs more explanation

Short signature strings are convenient for simple tasks. For a task that needs clearer instructions, you can define a Python class that inherits from `dspy.Signature`.

In a class-based signature:

- the class docstring explains the whole task;
- `dspy.InputField()` marks an input;
- `dspy.OutputField()` marks an output; and
- `desc=` adds a helpful note about one field.

The example also uses `Literal` to list the only allowed output values.


In [ ]:
from typing import Literal

class Emotion(dspy.Signature):
    """Classify the dominant emotion in the text."""

    text: str = dspy.InputField()
    emotion: Literal["joy", "anger", "fear", "surprise"] = dspy.OutputField(
        desc="exactly one label, the single strongest emotion"
    )

dspy.Predict(Emotion)(text="The lens room door was open. It is never open.")

> **Expected result:** `Prediction(emotion='fear')`
>
> The exact answer can depend on the model. Because the output type is `Literal["joy", "anger", "fear", "surprise"]`, DSPy asks for one of those four labels. This makes the result easier for other Python code to use.


DSPy keeps a signature's instructions and field information as data that your Python program can inspect.

The next cell prints that information. It also creates a new version of the signature with different instructions.


In [ ]:
print(Emotion.instructions)
print(Emotion.input_fields)
print(Emotion.output_fields)

Stricter = Emotion.with_instructions(
    "Classify the dominant emotion. If two emotions compete, prefer the one driving the final sentence."
)
print(Stricter.instructions)

> **What to look for:**
>
> - `Emotion.instructions` contains the class docstring.
> - `Emotion.input_fields` and `Emotion.output_fields` describe the fields.
> - `Stricter.instructions` contains the replacement instruction.

`with_instructions(...)` returns a new signature; it does not change `Emotion` in place. This matters later because DSPy optimizers can try different instructions for you. Notebooks 06 and 07 show that process.


## String and class syntax describe the same thing

We have now written signatures in two ways:

```python
dspy.Signature("question -> answer")
```

and:

```python
class MySignature(dspy.Signature):
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()
```

These are not two different DSPy features. The short string form builds the same kind of signature class for you and adds default instructions based on the field names.

Use a string for a quick, simple task. Use a class when you want to add task instructions or field descriptions.


In [ ]:
sig = dspy.Signature("question -> answer")       # the string form builds a class...
print(type(sig) is type(Emotion), issubclass(sig, dspy.Signature))

print(sig.instructions)                          # ...with default instructions filled in

# The bridge between the spellings: a string signature plus instructions
# IS a class signature — this is equivalent to writing the class by hand.
OneWord = dspy.Signature("question -> answer").with_instructions(
    "Answer in exactly one word."
)
print(OneWord.instructions)

> **Expected result:**
>
> - `True True`, showing that both signatures are the same kind of object;
> - default instructions similar to `Given the fields 'question', produce the fields 'answer'.`; and
> - `Answer in exactly one word.` for `OneWord`.

The important idea is that either syntax produces a signature with an `instructions` value. DSPy optimizers can update that value regardless of which syntax you started with.


## Two useful notes

1. **A model can return the wrong format.** If DSPy cannot convert a reply to the types in your signature, it raises `AdapterParseError`. This is uncommon with capable models but can happen more often with small local models. Notebook 02 introduces `JSONAdapter`, which can make structured results more reliable.
2. **Some older tutorials use outdated APIs.** In DSPy 3.x, the old `prefix=` and `format=` arguments for `InputField` do nothing; use `desc=` instead. You also do not need a separate `TypedPredictor`. Regular `dspy.Predict` supports typed outputs.


## Recap

| Term | Beginner-friendly meaning |
|---|---|
| signature | a description of the inputs and outputs of a model task |
| class-based signature | a longer form that can include task instructions and field notes |
| `dspy.Predict(signature)` | a callable module that runs the task |
| `dspy.Prediction` | the returned object; read outputs with attributes such as `pred.answer` |
| `with_instructions(...)` | creates a new signature with different instructions |
| `dspy.inspect_history(n)` | shows recent prompts and raw model replies |

The main idea is simple: describe **what** goes in and **what** should come out. Let DSPy handle much of the prompt formatting and response parsing.


## Exercises

1. Create a spam checker:

   ```python
   spam_checker = dspy.Predict("email -> is_spam: bool, reason: str")
   ```

   Try it with an obvious spam email and a normal newsletter. Does each `reason` match its `is_spam` value?

2. Rewrite the spam checker as a class-based signature. Add a docstring that tells the model to choose “not spam” when the evidence is unclear. Run both versions on the same newsletter, and then call `dspy.inspect_history(n=2)`. Can you find the different instructions?

3. Try this intentionally mismatched type:

   ```python
   dspy.Predict("question -> answer: int")(question="What color is the sky?")
   ```

   What happens when the task asks for a word but the signature requires an integer? Why is an error safer than silently returning the wrong Python type?

**Next up:** Notebook 02 explains how DSPy builds the prompt and introduces the two values that a predictor can learn.
